## Train Model

In this notebook, we will train our segmentation network using PyTorch and a UNet model.

---

### 1. Importing Libraries

We import the necessary libraries for our segmentation task:

In [1]:
import os
from torch.utils.data import DataLoader
import imgaug.augmenters as iaa
import torch
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

from dataset_seg import get_image_and_label_paths, ImgSegDataset
from model_seg import UNet

---
### 2. Defining  Paths

Here, we define the directories for the training and validation datasets.

In [2]:
root_dir = "data/processed/preprocessed_ircadb_liver_dataset"

train_dir = os.path.join(root_dir, "train")
val_dir = os.path.join(root_dir, "val")

---
### 3. Extracting Image and Label Paths

This function collects all image slices and their corresponding label files from the directories provided.

In [3]:
# Get image and label paths for training and validation datasets
train_image_paths, train_label_paths = get_image_and_label_paths(train_dir)
val_image_paths, val_label_paths = get_image_and_label_paths(val_dir)

print(f"Training dataset: {len(train_image_paths)} images, {len(train_label_paths)} labels")
print(f"Validation dataset: {len(val_image_paths)} images, {len(val_label_paths)} labels\n")

Training dataset: 2241 images, 2241 labels
Validation dataset: 582 images, 582 labels



---
### 4. Define Augmentation Pipeline

We define a data augmentation pipeline using `ImgAug` to apply scaling, rotation, and elastic transformations.

In [5]:
aug_seq = iaa.Sequential([
    iaa.Affine(scale=(0.85, 1.15), rotate=(-45, 45)),
    iaa.ElasticTransformation()
])

---
### 5. Creating Dataset Objects

Now, we create training and validation datasets with the defined augmentation pipeline applied only to the training set.

In [6]:
dataset_train = ImgSegDataset(train_image_paths, train_label_paths, aug_seq)
dataset_val = ImgSegDataset(val_image_paths, val_label_paths, None)

print(f"Length of training dataset: {len(dataset_train)}")
print(f"Length of validation dataset: {len(dataset_val)}")

Length of training dataset: 2241
Length of validation dataset: 582


---
### 6. Define Model Parameters

We define the training parameters such as number of epochs, batch size, and learning rate.

In [7]:
num_epochs = 20
batch_size = 8
learning_rate = 1e-4

---
### 7. Define Device

We check if a GPU is available and define the device to use for training.

In [8]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


---
### 8. Create DataLoaders

Now, we set up DataLoaders for training and validation datasets.

In [9]:
train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)

print("Number of batches in training set:", len(train_loader))
print("Number of batches in validation set:", len(val_loader))

Number of batches in training set: 281
Number of batches in validation set: 73


---
### 9. Custom Loss Function

We will combine Binary Cross-Entropy (BCE) with Dice Loss for a more robust segmentation loss function.

The Dice Loss is defined as:
$$ L(\hat{y}, y) = 1 - \frac{2 |\hat{y} \cap y|}{|\hat{y}| + |y|} $$

The intersection is easily computed as $\hat{y} * y$, where $\hat{y}$ and $y$ are binary masks.<br>
You can read more about the Dice Score here:
https://en.wikipedia.org/wiki/S%C3%B8rensen%E2%80%93Dice_coefficient


The **Dice score** is a common metric used in segmentation tasks. It measures how well the predicted mask matches the ground truth mask. The score ranges from 0 to 1, where 1 means a perfect match.

- **pred**: The predicted output from the model (after thresholding)
- **target**: The ground truth label (actual mask)
- **smooth**: A small value to avoid division by zero

In [10]:
# Define Dice Score function
def dice_coefficient(pred, target, smooth=1e-8):
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    dice_score = (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)
    return dice_score

# Binary Cross-Entropy with Dice Loss class
class BCEDiceLoss(torch.nn.Module):
    """Combines Binary Cross-Entropy (BCE) and Dice Loss for binary segmentation."""
    def __init__(self, weight_ce=1.0, weight_dice=1.0, smooth=1e-8):
        super().__init__()
        self.weight_ce = max(weight_ce, 0.01)  # Ensure non-zero weights
        self.weight_dice = max(weight_dice, 0.01)
        self.smooth = smooth

    def forward(self, pred, mask):
        # Binary Cross-Entropy Loss
        bce_loss = F.binary_cross_entropy(pred, mask)

        # Dice Loss computation
        dice_score = dice_coefficient(pred, mask, self.smooth)
        dice_loss = 1 - dice_score

        # Combined loss
        loss = self.weight_ce * bce_loss + self.weight_dice * dice_loss

        return loss

---
### 10. Initialize Model

We initialize the UNet model, loss function, and optimizer for training.

In [11]:
model = UNet().to(device)
loss_fn = BCEDiceLoss(weight_ce=1.0, weight_dice=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, amsgrad=True)

---
### 11. Setting Up TensorBoard for Logging

We use TensorBoard to log training and validation losses during the training process.

#### How to Visualize Loss Functions in TensorBoard:

1. Open Command Prompt (cmd) and navigate to the parent directory of the `runs` folder.

2. Activate your environment in cmd using: conda activate `your_env`

3. Run one of the following commands:

    tensorboard --logdir=runs

or 

        python -m tensorboard.main --logdir=runs

4. Open a web browser and go to:
http://localhost:6006/

In [12]:
writer = SummaryWriter()

---
### 12. Training Loop

We define the main training loop that performs forward passes, calculates losses, backpropagates, and updates the model weights.

In [13]:
for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss_train = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Skip smaller batches during training
        if inputs.size(0) < batch_size:
            continue

        predictions = model(inputs)

        # Calculate the loss
        loss = loss_fn(predictions, targets)

        # Backpropagation and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate training loss
        running_loss_train += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.4f}")

    # Compute average epoch-level loss
    epoch_loss_train = running_loss_train / len(train_loader)

    # Log to TensorBoard
    writer.add_scalar('Loss/Train_Epoch', epoch_loss_train, epoch)

    # Validation phase
    model.eval()
    running_loss_val = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(val_loader):
            inputs = inputs.to(device)
            targets = targets.to(device)

            predictions = model(inputs)
            loss = loss_fn(predictions, targets)

            # Accumulate validation loss
            running_loss_val += loss.item()

    # Compute average epoch-level validation loss
    epoch_loss_val = running_loss_val / len(val_loader)
    print(f"\t\t\t\t ===> Validation Loss: {epoch_loss_val:.4f}")

    # Log to TensorBoard
    writer.add_scalar('Loss/Val_Epoch', epoch_loss_val, epoch)

writer.close()

Epoch: 0, Batch: 0, Loss: 1.5025
Epoch: 0, Batch: 50, Loss: 1.0339
Epoch: 0, Batch: 100, Loss: 1.0064
Epoch: 0, Batch: 150, Loss: 0.8622
Epoch: 0, Batch: 200, Loss: 0.8029
Epoch: 0, Batch: 250, Loss: 0.5249
				 ===> Validation Loss: 0.8983
Epoch: 1, Batch: 0, Loss: 0.6182
Epoch: 1, Batch: 50, Loss: 0.8102
Epoch: 1, Batch: 100, Loss: 0.5695
Epoch: 1, Batch: 150, Loss: 0.7284
Epoch: 1, Batch: 200, Loss: 0.3429
Epoch: 1, Batch: 250, Loss: 0.7735
				 ===> Validation Loss: 1.1659
Epoch: 2, Batch: 0, Loss: 0.6249
Epoch: 2, Batch: 50, Loss: 0.4019
Epoch: 2, Batch: 100, Loss: 0.5020
Epoch: 2, Batch: 150, Loss: 0.5741
Epoch: 2, Batch: 200, Loss: 0.6895
Epoch: 2, Batch: 250, Loss: 0.6234
				 ===> Validation Loss: 0.7726
Epoch: 3, Batch: 0, Loss: 0.3663
Epoch: 3, Batch: 50, Loss: 0.8676
Epoch: 3, Batch: 100, Loss: 0.4270
Epoch: 3, Batch: 150, Loss: 0.5051
Epoch: 3, Batch: 200, Loss: 0.6854
Epoch: 3, Batch: 250, Loss: 0.8469
				 ===> Validation Loss: 0.9286
Epoch: 4, Batch: 0, Loss: 0.9775
Epo

---
### 13. Save Model Weights

After training, we save the model's weights to a file for using in the evaluation phase of the segmentation model.

In [ ]:
weights_path = os.path.join("data", "model_weights", f"seg_liver_epoch{epoch + 1}_task.pth")

torch.save(model.state_dict(), weights_path)